In [3]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch; from torch import nn; from d2l import torch as d2l
if os.path.basename(os.getcwd()) == 'notes': os.chdir('..')
print('工作目录:', os.getcwd())

工作目录: d:\github\prediction


In [ ]:
# ============================================================
# 从 NetCDF 读取 #0 站点 → 80/15/5 分割 → 预处理 → MLP 直接多步训练 (48步)
# 先分割再插值, 杜绝边界数据泄漏; pm_ave 保持原始量纲
# 直接多步: 过去 tau=48 步 → 一次输出未来 H=48 步 (不递归, 无误差累积)
# ============================================================
import netCDF4 as nc

def load_station(nc_path, idx=0):
    """从 NetCDF 读取指定站点, 转为带 datetime 索引的 DataFrame"""
    f = nc.Dataset(nc_path)
    t = f.variables['time'][:]
    dt = pd.Timestamp(f.variables['time'].units.split('since ')[1]) + pd.to_timedelta(t, unit='h')
    pm25 = f.variables['PM2.5'][:, idx]
    f.close()
    df = pd.DataFrame({'PM_Jingan': pm25, 'PM_US Post': pm25, 'PM_Xuhui': pm25}, index=dt)
    df.index.name = 'datetime'
    return df.resample('1h').first()

def split_and_preprocess(df, ratios=(0.80, 0.15, 0.05)):
    """分割 → 各子集独立预处理 → 返回 train/val/test"""
    n = len(df); n1 = int(n * ratios[0]); n2 = int(n * sum(ratios[:2]))
    subs = []
    for raw in [df.iloc[:n1], df.iloc[n1:n2], df.iloc[n2:]]:
        d = raw.copy()
        d['pm_ave'] = d[['PM_Jingan','PM_US Post','PM_Xuhui']].mean(axis=1)
        d['pm_ave'] = d['pm_ave'].interpolate(method='time', limit=168).ffill().bfill()
        subs.append(d)
    return subs

df_raw = load_station('data3/dataset_yrd.nc', idx=0)
df_tr, df_va, df_te = split_and_preprocess(df_raw)
print(f'训练 {len(df_tr)} | 验证 {len(df_va)} | 测试 {len(df_te)}')

# pm_ave 提取 + 标准化 (用训练集统计量)
x_tr = torch.tensor(df_tr['pm_ave'].values, dtype=torch.float32)
x_va = torch.tensor(df_va['pm_ave'].values, dtype=torch.float32)
x_te = torch.tensor(df_te['pm_ave'].values, dtype=torch.float32)
mean, std = x_tr.mean(), x_tr.std()
x_tr, x_va, x_te = (x_tr - mean) / std, (x_va - mean) / std, (x_te - mean) / std

# 直接多步特征 (向量化): 过去 tau 步 → 未来 H 步
tau, H = 48, 48
def make_feats(x, tau, H):
    n = len(x) - tau - H + 1
    feats = torch.stack([x[i:i+n] for i in range(tau)], dim=1)        # (n, tau) 输入
    labs  = torch.stack([x[tau+k:tau+k+n] for k in range(H)], dim=1)  # (n, H) 未来 H 步
    return feats, labs

ftr_tr, lab_tr = make_feats(x_tr, tau, H)
ftr_va, lab_va = make_feats(x_va, tau, H)
print(f'直接多步样本: 训练 {tuple(ftr_tr.shape)}→{tuple(lab_tr.shape)} | 验证 {tuple(ftr_va.shape)}')
train_iter = d2l.load_array((ftr_tr, lab_tr), 16, is_train=True)
val_iter   = d2l.load_array((ftr_va, lab_va), 16, is_train=False)

# MLP: Linear(tau,128) → ReLU → Linear(128,H)
def get_net():
    net = nn.Sequential(nn.Linear(tau, 128), nn.ReLU(), nn.Linear(128, H))
    net.apply(lambda m: nn.init.xavier_uniform_(m.weight) if type(m)==nn.Linear else None)
    return net

loss = nn.MSELoss(reduction='none')
def train(net, tr_it, va_it, loss, epochs, lr, wd=0.002):
    opt = torch.optim.Adam(net.parameters(), lr)
    for ep in range(epochs):
        net.train()
        for X, y in tr_it:
            opt.zero_grad()
            l = loss(net(X), y); l2 = sum((p**2).sum() for p in net.parameters())
            (l.sum() + wd * l2).backward(); opt.step()
        print(f'epoch {ep+1}, train {d2l.evaluate_loss(net,tr_it,loss):.4f}, val {d2l.evaluate_loss(net,va_it,loss):.4f}')

net = get_net()
train(net, train_iter, val_iter, loss, 20, 0.01)

In [ ]:
# ============================================================
# 36 组评估: 每组 tau=48h 上下文 → MLP 一次预测 48h, RMSE (ug/m3)
# 同时报告 48h RMSE 与前 24h RMSE (参考)
# ============================================================
N_GROUPS, N_CTX, N_PRED = 36, tau, H
STRIDE = (len(df_te) - N_CTX - N_PRED) // (N_GROUPS - 1)
actual_all = df_te['pm_ave'].values

rmses, rmses24 = [], []
fig, axes = plt.subplots(6, 6, figsize=(36, 36)); axes = axes.flatten()
for g in range(N_GROUPS):
    s = g * STRIDE
    w = x_te[s:s+tau].reshape(1, -1)                    # (1, tau) 标准化上下文
    with torch.no_grad():
        pred_std = net(w).numpy().reshape(-1)          # (H,) 直接多步输出
    preds = pred_std * std.item() + mean.item()        # 反标准化
    actual = actual_all[s+N_CTX:s+N_CTX+N_PRED]
    err2 = (preds - actual) ** 2
    rmses.append(np.sqrt(err2.mean()))                 # 48h RMSE
    rmses24.append(np.sqrt(err2[:24].mean()))          # 前 24h RMSE (参考)
    ax = axes[g]; h = np.arange(1, N_PRED+1)
    ax.plot(h, actual, lw=0.8, label='actual')
    ax.plot(h, preds, lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[-1]:.1f}', fontsize=9)
    ax.legend(fontsize=7); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'MLP Direct — {N_GROUPS} Groups (RMSE 48h)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()
rmses = np.array(rmses); rmses24 = np.array(rmses24)
print(f'\n=== {N_GROUPS} 组 RMSE 汇总 (48h | 24h) ===')
for g in range(N_GROUPS): print(f'  G{g+1:2d}: {rmses[g]:6.2f} | {rmses24[g]:6.2f}')
print(f'\n平均 RMSE(48h) = {rmses.mean():.2f} | 参考 RMSE(24h) = {rmses24.mean():.2f}')